Load data

In [1]:
import pandas as pd
import numpy as np
import json

def load_data(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path.endswith(".csv"):
        return pd.read_csv(path)
    else:
        raise ValueError("Unsupported format")

taf_df = load_data("taf_2025.csv")
metar_df = load_data("metar_2025.csv")
flights_df = load_data("klm_cleaned_data_2025.parquet")

In [4]:
# Legacy METAR feature columns (JSON-encoded format — not used with taf_2025)
METAR_FEATURES = [
    "rowkey",
    "airport",
    "date",
    "hour",
    "minute",
    "timestamp",
    "actual_metar__wind_speed",
    "actual_metar__wind_degrees",
    "actual_metar__wind_gust",
    "actual_metar__visibility_value"
    # "actual_metar__pressure_value",
    # "actual_metar__conditions_json",
    # "actual_metar__clouds_json",
    # "actual_metar__is_auto"
]

In [3]:
# Flat TAF feature columns from taf_2025 (CSV / Parquet)
TAF_FEATURES = [
    "rowkey",
    "airport",
    "date",
    "hour",
    "minute",
    "timestamp",
    "wind_speed",
    "wind_degrees",
    "wind_gust",
    "wind_probability",
    "visibility",
    "rainfall_average",
    "rainfall_min",
    "rainfall_max",
    "thunderstorm",
    "supercooled_precipitation",
    "snow",
    "wind_chill",
    "wind_speed_extra"
]

Clean nulls

In [5]:
def clean_fake_nulls(df):
    fake_nulls = ["null", "None", "", "[]", "NaN"]
    return df.replace(fake_nulls, np.nan)

taf_df = clean_fake_nulls(taf_df)
metar_df = clean_fake_nulls(metar_df)
flights_df = clean_fake_nulls(flights_df)

In [7]:
def basic_eda(df, name="dataset"):
    print(f"\n--- {name} ---")
    print("Shape:", df.shape)
    print("\nMissing values (%):")
    print((df.isnull().mean() * 100).sort_values(ascending=False).head(20))
    print("\nDtypes:")
    print(df.dtypes)

# basic_eda(taf_df, "TAF 2025")
basic_eda(metar_df, "METAR 2025")


--- METAR 2025 ---
Shape: (17485, 10)

Missing values (%):
minute          100.000000
wind_gust        98.141264
wind_degrees      2.533600
rowkey            0.000000
airport           0.000000
date              0.000000
timestamp         0.000000
hour              0.000000
wind_speed        0.000000
visibility        0.000000
dtype: float64

Dtypes:
rowkey              str
airport             str
date                str
hour              int64
minute          float64
timestamp           str
wind_speed      float64
wind_degrees    float64
wind_gust       float64
visibility      float64
dtype: object


In [10]:
print(taf_df.shape)
print(taf_df.dtypes)
taf_df.describe()

(53989, 21)
rowkey                           str
airport                          str
date                             str
hour                           int64
minute                       float64
timestamp                        str
wind_speed                   float64
wind_degrees                 float64
wind_gust                    float64
wind_probability             float64
visibility                   float64
clouds_probability           float64
conditions_probability       float64
rainfall_average             float64
rainfall_min                 float64
rainfall_max                 float64
thunderstorm                 float64
supercooled_precipitation    float64
snow                         float64
wind_chill                   float64
wind_speed_extra             float64
dtype: object


,hour,minute,wind_speed,wind_degrees,wind_gust,wind_probability,visibility,clouds_probability,conditions_probability,rainfall_average,rainfall_min,rainfall_max,thunderstorm,supercooled_precipitation,snow,wind_chill,wind_speed_extra
count,53989.000000,49788.000000,1166.000000,1115.000000,102.000000,47.000000,1166.000000,352.000000,313.000000,49788.000000,49788.000000,49788.000000,3035.000000,3035.000000,3035.000000,3035.000000,3035.000000
mean,11.490785,26.270788,9.548885,195.874439,29.352941,24.893617,1665.686106,25.568182,24.984026,0.048545,0.029156,0.078012,0.443163,0.004942,0.209226,-47.839209,9.477100
std,6.920626,17.183329,4.617629,95.975678,4.288246,15.304495,2596.710365,15.496196,15.732634,0.345609,0.203519,0.643316,2.099922,0.202917,2.370032,50.631853,4.662829
min,0.000000,0.000000,2.000000,10.000000,25.000000,0.000000,10.000000,0.000000,0.000000,0.000392,0.000392,0.000392,0.000000,0.000000,0.000000,-99.000000,1.000000
25%,6.000000,10.000000,6.000000,120.000000,25.000000,15.000000,10.000000,0.000000,0.000000,0.000392,0.000392,0.000392,0.000000,0.000000,0.000000,-99.000000,6.000000
50%,11.000000,25.000000,8.000000,200.000000,29.000000,30.000000,10.000000,30.000000,30.000000,0.000392,0.000392,0.000392,0.000000,0.000000,0.000000,-8.000000,9.000000
75%,17.000000,40.000000,12.000000,270.000000,31.750000,35.000000,4000.000000,40.000000,40.000000,0.000392,0.000392,0.000392,0.000000,0.000000,0.000000,2.000000,13.000000
max,23.000000,55.000000,28.000000,360.000000,42.000000,40.000000,9000.000000,40.000000,40.000000,28.999804,15.399265,52.329911,20.000000,10.000000,80.000000,10.000000,32.000000


In [11]:
print(metar_df.shape)
print(metar_df.dtypes)
metar_df.describe()

(17485, 10)
rowkey              str
airport             str
date                str
hour              int64
minute          float64
timestamp           str
wind_speed      float64
wind_degrees    float64
wind_gust       float64
visibility      float64
dtype: object


,hour,minute,wind_speed,wind_degrees,wind_gust,visibility
count,17485.000000,0.0,17485.000000,17042.000000,325.000000,17485.000000
mean,11.498256,NaN,8.972434,187.905175,31.803077,875.694023
std,6.923699,NaN,4.847920,96.920488,5.314027,2306.697627
min,0.000000,NaN,0.000000,0.000000,15.000000,10.000000
25%,5.000000,NaN,5.000000,100.000000,28.000000,10.000000
50%,12.000000,NaN,8.000000,190.000000,31.000000,10.000000
75%,17.000000,NaN,12.000000,270.000000,35.000000,10.000000
max,23.000000,NaN,36.000000,360.000000,49.000000,9000.000000


In [ ]:
def parse_taf_timestamps(df):
    """
    Construct forecast valid_time from date + hour columns,
    and parse the TAF issue_time from the timestamp column.
    Works on both CSV and Parquet inputs with the taf_2025 schema.
    """
    df = df.copy()
    hour_str = df["hour"].fillna(0).astype(int).astype(str).str.zfill(2)
    df["valid_time"] = pd.to_datetime(
        df["date"].astype(str) + " " + hour_str + ":00:00",
        errors="coerce"
    )
    df["issue_time"] = pd.to_datetime(
        df["timestamp"].astype(str).str.replace(" UTC", "", regex=False),
        errors="coerce"
    )
    return df

taf_df = parse_taf_timestamps(taf_df)


In [ ]:
taf_df.head()

In [ ]:
def prepare_taf_features(df, features):
    """
    Ensure all TAF weather feature columns are numeric.
    Works on both CSV and Parquet inputs with the taf_2025 schema.
    """
    df = df.copy()
    for col in features:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

taf_df = prepare_taf_features(taf_df, TAF_FEATURES)

In [ ]:
print(taf_df[["valid_time", "issue_time"] + TAF_FEATURES].head())

In [ ]:
flights_df["arrival_ts"] = pd.to_datetime(
    flights_df["fl_scheduled_in_block_time_utc"], errors="coerce"
)

flights_df = flights_df.sort_values("arrival_ts")

In [ ]:
def build_taf_features(taf_df, features=None):
    """
    Return a sorted DataFrame of TAF feature columns ready for asof merging.
    Works on both CSV and Parquet inputs with the taf_2025 schema.
    """
    if features is None:
        features = TAF_FEATURES
    cols = ["valid_time"] + [f for f in features if f in taf_df.columns]
    return taf_df[cols].sort_values("valid_time").dropna(subset=["valid_time"])


In [ ]:
taf_features_df = build_taf_features(taf_df)

print(taf_features_df.shape)
print(taf_features_df.head())

In [ ]:
# TAF data is already flat — taf_features_df contains all forecast features.
# No separate TAF extraction step needed (unlike JSON-encoded METAR format).
print(taf_features_df.columns.tolist())


In [ ]:
def attach_taf_to_flights(flights_df, taf_df, features=None):
    """
    Merge TAF forecasts onto flights using an asof join on valid_time ↔ arrival_ts.
    Matches each flight to the most recent TAF forecast period before its arrival.
    Works on both CSV and Parquet inputs with the taf_2025 schema.
    """
    if features is None:
        features = TAF_FEATURES

    flights = flights_df.copy()
    flights["arrival_ts"] = pd.to_datetime(
        flights["fl_scheduled_in_block_time_utc"], errors="coerce"
    )
    flights = flights.sort_values("arrival_ts")

    cols = ["valid_time"] + [f for f in features if f in taf_df.columns]
    taf_sorted = taf_df[cols].sort_values("valid_time").dropna(subset=["valid_time"])

    merged = pd.merge_asof(
        flights,
        taf_sorted,
        left_on="arrival_ts",
        right_on="valid_time",
        direction="backward"
    )
    return merged

In [ ]:
flights_taf = attach_taf_to_flights(flights_df, taf_df)

print(flights_taf.shape)
flights_taf.head()

In [ ]:
flights_taf["wind_dir_rad"] = np.deg2rad(flights_taf["wind_degrees"])

flights_taf["wind_sin_taf"] = np.sin(flights_taf["wind_dir_rad"])
flights_taf["wind_cos_taf"] = np.cos(flights_taf["wind_dir_rad"])

In [ ]:
def runway_to_heading(runway):
    try:
        return int(str(runway)[:2]) * 10
    except:
        return np.nan

flights_taf["runway_heading"] = flights_taf["fp_planned_arrival_runway"].apply(runway_to_heading)

In [ ]:
angle_diff = np.deg2rad(
    flights_taf["wind_degrees"] - flights_taf["runway_heading"]
)

flights_taf["wind_along_runway_kt"] = (
    flights_taf["wind_speed"] * np.cos(angle_diff)
)

flights_taf["crosswind_signed_kt"] = (
    flights_taf["wind_speed"] * np.sin(angle_diff)
)

flights_taf["crosswind_abs_kt"] = np.abs(
    flights_taf["crosswind_signed_kt"]
)

In [ ]:
def parse_visibility(val):
    if pd.isna(val):
        return np.nan

    val = str(val)

    try:
        if "SM" in val:
            return float(val.replace("SM", "")) * 1609.34
        return float(val)
    except:
        return np.nan

flights_taf["visibility_m"] = flights_taf["visibility"].apply(parse_visibility)

In [ ]:
def build_weather_pipeline(taf_df, feature_map):
    """
    Map TAF column names to output names, returning a dict of DataFrames.
    Works on both CSV and Parquet inputs with the taf_2025 schema.
    """
    outputs = {}
    for col, name in feature_map.items():
        if col in taf_df.columns:
            outputs[name] = taf_df[["valid_time", col]].rename(columns={col: name})
    return outputs

taf_features = build_weather_pipeline(taf_df, TAF_FEATURES)  #TODO: check?


In [ ]:
taf_features = build_weather_pipeline(taf_df, {
    "wind_speed": "wind_speed",
    "wind_degrees": "wind_dir",
    "visibility": "visibility"
})

JSON exploder

In [ ]:
def build_taf_long(taf_df, features=None):
    """
    Convert flat TAF DataFrame to long format for per-feature analysis.
    Works on both CSV and Parquet inputs with the taf_2025 schema.
    """
    if features is None:
        features = TAF_FEATURES

    rows = []
    for col in features:
        if col not in taf_df.columns:
            continue
        subset = taf_df[["rowkey", "valid_time", col]].copy()
        subset = subset.rename(columns={col: "value"})
        subset["feature"] = col
        rows.append(subset)

    if not rows:
        return pd.DataFrame(columns=["rowkey", "valid_time", "feature", "value"])

    long_df = pd.concat(rows, ignore_index=True)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    return long_df.dropna(subset=["valid_time"])


In [ ]:
taf_long = build_taf_long(taf_df)

print(taf_long.head())
print(taf_long.columns.tolist())
print(taf_long.shape)

In [ ]:
taf_long["valid_time"] = pd.to_datetime(taf_long["valid_time"], errors="coerce")
taf_long["value"] = pd.to_numeric(taf_long["value"], errors="coerce")
taf_long = taf_long.dropna(subset=["valid_time"])

taf_long["valid_time"].describe()